# Retail Sales Performance Analysis

A portfolio project focused on data cleaning, exploratory data analysis, profitability, sales trends, and business KPIs.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# --- Visual style: one consistent theme, but a distinct color role per chart type,
# so ranked bars, category bars, trend lines, and comparisons don't all look the same ---
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titlepad'] = 12
plt.rcParams['figure.facecolor'] = 'white'

CATEGORICAL = sns.color_palette("Set2")                  # distinct hues: unordered categories
SEQUENTIAL  = sns.color_palette("crest", as_cmap=True)    # gradient: ranked / sorted bars
ACCENT_A    = "#2E6F95"   # primary trend line
ACCENT_B    = "#E07A5F"   # secondary trend line
GOOD_BAD    = ["#588157", "#BC4749"]   # semantic pair: positive vs negative outcome

def add_bar_labels(ax, fmt="{:,.0f}"):
    for p in ax.patches:
        h = p.get_height()
        if pd.notna(h):
            ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2, h),
                        ha='center', va='bottom', fontsize=9, color='#333333',
                        xytext=(0, 3), textcoords='offset points')

## 2. Load Dataset

In [4]:
df = pd.read_csv("retail_sales_data.csv")
df.head()

,Order_ID,Order_Date,Region,Category,Product,Quantity,Unit_Price,Discount,Sales,Cost
0,100001,2024-06-02,West,Electronics,Smartphone,4,274.16,0.10,986.99,554.85
1,100002,2025-05-29,South,Sports,Dumbbells,4,106.15,0.00,424.60,312.22
2,100003,2024-05-20,East,Beauty,Skincare Set,2,68.36,0.05,129.88,90.00
3,100004,2024-01-16,East,Home,Cookware,5,90.31,0.00,451.55,357.20
4,100005,2024-05-11,South,Home,Cookware,1,90.89,0.10,81.80,64.64


## 3. Data Understanding

In [5]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Shape: (6120, 10)

Data types:
Order_ID        int64
Order_Date     object
Region         object
Category       object
Product        object
Quantity        int64
Unit_Price    float64
Discount      float64
Sales         float64
Cost          float64
dtype: object

Missing values:
Order_ID       0
Order_Date     0
Region        93
Category      63
Product        0
Quantity       0
Unit_Price    61
Discount       0
Sales         74
Cost           0
dtype: int64

Duplicate rows: 120


In [6]:
df.describe(include='all')

,Order_ID,Order_Date,Region,Category,Product,Quantity,Unit_Price,Discount,Sales,Cost
count,6120.000000,6120,6027,6057,6120,6120.000000,6059.000000,6120.000000,6046.000000,6120.000000
unique,NaN,547,4,5,20,NaN,NaN,NaN,NaN,NaN
top,NaN,2025-01-10,North,Electronics,Smartphone,NaN,NaN,NaN,NaN,NaN
freq,NaN,24,1555,1390,378,NaN,NaN,NaN,NaN,NaN
mean,102942.852941,NaN,NaN,NaN,NaN,2.981863,143.326554,0.070065,392.741318,271.490708
std,1762.914322,NaN,NaN,NaN,NaN,1.446829,125.924307,0.062674,427.516126,294.458587
min,100001.000000,NaN,NaN,NaN,NaN,-3.000000,41.270000,0.000000,-496.040000,20.280000
25%,101410.750000,NaN,NaN,NaN,NaN,2.000000,64.630000,0.000000,137.782500,93.735000
50%,102940.500000,NaN,NaN,NaN,NaN,3.000000,83.710000,0.050000,248.920000,168.745000
75%,104470.250000,NaN,NaN,NaN,NaN,4.000000,121.835000,0.100000,411.152500,285.607500


## 4. Data Cleaning

In [7]:
df = df.drop_duplicates().reset_index(drop=True)

for col in ['Region','Category','Product']:
    df[col] = df[col].astype('string').str.strip().str.title()

df['Region'] = df['Region'].fillna('Unknown')
df['Category'] = df['Category'].fillna('Unknown')
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')

df.loc[df['Quantity'] <= 0, 'Quantity'] = np.nan
df.loc[df['Unit_Price'] <= 0, 'Unit_Price'] = np.nan
df.loc[df['Sales'] <= 0, 'Sales'] = np.nan

for col in ['Quantity','Unit_Price','Sales']:
    df[col] = df[col].fillna(df[col].median())

df['Profit'] = df['Sales'] - df['Cost']
df['Profit_Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

print(df.isnull().sum())

Order_ID         0
Order_Date       0
Region           0
Category         0
Product          0
Quantity         0
Unit_Price       0
Discount         0
Sales            0
Cost             0
Profit           0
Profit_Margin    0
dtype: int64


## 5. Feature Engineering

In [8]:
df['Month'] = df['Order_Date'].dt.to_period('M').astype(str)
df['Quarter'] = df['Order_Date'].dt.quarter
df['Sales_Band'] = pd.cut(
    df['Sales'],
    bins=[0,100,500,1000,2500,5000,np.inf],
    labels=['0-100','100-500','500-1K','1K-2.5K','2.5K-5K','5K+']
)
df.head()

,Order_ID,Order_Date,Region,Category,Product,Quantity,Unit_Price,Discount,Sales,Cost,Profit,Profit_Margin,Month,Quarter,Sales_Band
0,100001,2024-06-02,West,Electronics,Smartphone,4.0,274.16,0.10,986.99,554.85,432.14,43.78,2024-06,2,500-1K
1,100002,2025-05-29,South,Sports,Dumbbells,4.0,106.15,0.00,424.60,312.22,112.38,26.47,2025-05,2,100-500
2,100003,2024-05-20,East,Beauty,Skincare Set,2.0,68.36,0.05,129.88,90.00,39.88,30.71,2024-05,2,100-500
3,100004,2024-01-16,East,Home,Cookware,5.0,90.31,0.00,451.55,357.20,94.35,20.89,2024-01,1,100-500
4,100005,2024-05-11,South,Home,Cookware,1.0,90.89,0.10,81.80,64.64,17.16,20.98,2024-05,2,0-100


## 6. Exploratory Data Analysis

In [ ]:
category_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)

colors = SEQUENTIAL(np.linspace(0.25, 0.9, len(category_sales)))
plt.figure(figsize=(8, 4.5))
ax = plt.gca()
ax.bar(category_sales.index, category_sales.values, color=colors, edgecolor='white', linewidth=0.6)
ax.set_title('Total Sales by Category')
ax.set_ylabel('Sales')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=20)
add_bar_labels(ax, "{:,.0f}")
plt.tight_layout()
plt.show()

In [ ]:
region_profit = df.groupby('Region')['Profit'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4.5))
ax = plt.gca()
ax.bar(region_profit.index, region_profit.values, color=CATEGORICAL[:len(region_profit)],
       edgecolor='white', linewidth=0.6)
ax.set_title('Total Profit by Region')
ax.set_ylabel('Profit')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
add_bar_labels(ax, "{:,.0f}")
plt.tight_layout()
plt.show()

In [ ]:
monthly_sales = df.groupby('Month')['Sales'].sum()

plt.figure(figsize=(11, 4.5))
ax = plt.gca()
ax.plot(monthly_sales.index, monthly_sales.values, marker='o', color=ACCENT_A,
        linewidth=2.2, markersize=5, markerfacecolor='white', markeredgewidth=1.5)
ax.fill_between(range(len(monthly_sales)), monthly_sales.values, color=ACCENT_A, alpha=0.12)
ax.set_title('Monthly Sales Trend')
ax.set_ylabel('Sales')
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
discount_margin = df.groupby('Discount')['Profit_Margin'].mean().sort_index()

x_labels = (discount_margin.index * 100).astype(str)
plt.figure(figsize=(7, 4.5))
ax = plt.gca()
ax.plot(x_labels, discount_margin.values, marker='o', color=ACCENT_B,
        linewidth=2.2, markersize=6, markerfacecolor='white', markeredgewidth=1.5)
ax.fill_between(range(len(x_labels)), discount_margin.values, color=ACCENT_B, alpha=0.12)
ax.set_title('Average Profit Margin by Discount')
ax.set_xlabel('Discount (%)')
ax.set_ylabel('Profit Margin (%)')
plt.tight_layout()
plt.show()

## 7. Statistical Analysis

In [13]:
no_discount = df.loc[df['Discount']==0, 'Profit']
discounted = df.loc[df['Discount']>0, 'Profit']

t_stat, p_value = stats.ttest_ind(no_discount, discounted, equal_var=False)

print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.5f}")
print("Statistically significant." if p_value < 0.05 else "Not statistically significant.")

T-statistic: 3.171
P-value: 0.00153
Statistically significant.


## 8. KPI Summary

In [14]:
kpis = pd.Series({
    'Total Sales': df['Sales'].sum(),
    'Total Profit': df['Profit'].sum(),
    'Average Order Value': df['Sales'].mean(),
    'Average Profit Margin (%)': df['Profit_Margin'].mean(),
    'Orders': df['Order_ID'].nunique()
})
kpis.round(2)

Total Sales                  2367685.40
Total Profit                  737857.60
Average Order Value              394.61
Average Profit Margin (%)         30.94
Orders                          6000.00
dtype: float64

## 9. Business Insights

- Identify the strongest sales categories and regions.
- Monitor monthly trends for growth and seasonality.
- Evaluate how discount levels relate to profitability.
- Use the KPIs to support commercial decision-making.

## 10. Executive Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('Retail Sales Analytics Dashboard', fontsize=18, fontweight='bold')

axes[0, 0].bar(category_sales.index, category_sales.values,
               color=SEQUENTIAL(np.linspace(0.25, 0.9, len(category_sales))), edgecolor='white', linewidth=0.6)
axes[0, 0].set_title('Sales by Category')

axes[0, 1].bar(region_profit.index, region_profit.values,
               color=CATEGORICAL[:len(region_profit)], edgecolor='white', linewidth=0.6)
axes[0, 1].set_title('Profit by Region')

axes[1, 0].plot(monthly_sales.index, monthly_sales.values, marker='o', color=ACCENT_A, linewidth=2.2,
                markersize=5, markerfacecolor='white', markeredgewidth=1.5)
axes[1, 0].fill_between(range(len(monthly_sales)), monthly_sales.values, color=ACCENT_A, alpha=0.12)
axes[1, 0].set_title('Monthly Sales Trend')
axes[1, 0].tick_params(axis='x', rotation=60)

axes[1, 1].plot((discount_margin.index * 100).astype(str), discount_margin.values, marker='o',
                color=ACCENT_B, linewidth=2.2, markersize=6, markerfacecolor='white', markeredgewidth=1.5)
axes[1, 1].fill_between(range(len(discount_margin)), discount_margin.values, color=ACCENT_B, alpha=0.12)
axes[1, 1].set_title('Profit Margin vs Discount')

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()